<a href="https://colab.research.google.com/github/MaazKhan53/ML-01-Run-the-Starter-Notebooks/blob/main/Copy_of_w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MaazKhan53/ML-01-Run-the-Starter-Notebooks/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

## The Rule:

The "High-Volume, Low-CTR Decay" baseline rule. We isolate and penalize pages that have secured a Page 1 Google ranking (Average Position <= 10) and generate significant visibility (Impressions >= 100), but fail to convert that visibility into traffic (CTR < 1%). The score is weighted logarithmically by impression volume to prioritize high-traffic bleeds.

## Reason Codes:

PAGE_1_LOW_CTR_BLEED: Triggered when Average Position <= 10, Impressions >= 100, and CTR < 0.01. Action: URGENT_METADATA_REFRESH.

STABLE_OR_LOW_PRIORITY: Triggered when the page maintains healthy CTR, sits off Page 1, or lacks statistical volume. Action: MONITOR

In [ ]:
import pandas as pd
import numpy as np
import os
from datasets import load_dataset
from google.colab import userdata

# Securely load token and warehouse data
hf_token = userdata.get('HF_TOKEN')
print("Connecting to warehouse and loading March 2026 partition...")
dataset = load_dataset(
    "FlyRank/internship-warehouse",
    data_files="fact_content_daily_performance/month=2026-03/*.parquet",
    split="train",
    token=hf_token
)
df = dataset.to_pandas()

# Contract enforcement: Drop rows without GA4 data
available_df = df[df['ga4_data_available'] == True].copy()
print(f"Usable rows after GA4 filter: {len(available_df):,}")

Connecting to warehouse and loading March 2026 partition...


README.md:   0%|          | 0.00/3.04k [00:00<?, ?B/s]

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B /  124MB            

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

Generating train split: 0 examples [00:00, ? examples/s]

Usable rows after GA4 filter: 413,966


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [ ]:
# Create baseline frame
baseline_df = available_df.copy()

# Calculate safe CTR
baseline_df['ctr'] = np.where(
    baseline_df['gsc_impressions'] > 0,
    baseline_df['gsc_clicks'] / baseline_df['gsc_impressions'],
    0
)

# 1. Calculate Score (Heuristic equation: log of impressions heavily penalized by CTR)
baseline_df['baseline_score'] = np.where(
    (baseline_df['gsc_avg_position'] <= 10) & (baseline_df['gsc_impressions'] >= 100),
    (np.log1p(baseline_df['gsc_impressions']) * 10) - (baseline_df['ctr'] * 100),
    0
)

# 2. Assign Reason Codes
baseline_df['reason_code'] = np.where(
    (baseline_df['baseline_score'] > 0) & (baseline_df['ctr'] < 0.01),
    'PAGE_1_LOW_CTR_BLEED',
    'STABLE_OR_LOW_PRIORITY'
)

# 3. Assign Actions
baseline_df['action_label'] = np.where(
    baseline_df['reason_code'] == 'PAGE_1_LOW_CTR_BLEED',
    'URGENT_METADATA_REFRESH',
    'MONITOR'
)

# 4. Rank the queue
ranked_queue = baseline_df[baseline_df['action_label'] == 'URGENT_METADATA_REFRESH'].sort_values(
    by='baseline_score', ascending=False
)[['content_hash_id', 'client_hash_id', 'gsc_impressions', 'gsc_avg_position', 'ctr', 'baseline_score', 'reason_code', 'action_label']]

# 5. Write to outputs directory
os.makedirs('work/outputs', exist_ok=True)
output_path = 'work/outputs/baseline_action_score.csv'
ranked_queue.to_csv(output_path, index=False)
print(f"Rule applied. Ranked queue ({len(ranked_queue)} rows) saved to: {output_path}")

print("\n--- TOP 20 RANKED QUEUE FOR REVIEW ---")
print(ranked_queue.head(20))

Rule applied. Ranked queue (71462 rows) saved to: work/outputs/baseline_action_score.csv

--- TOP 20 RANKED QUEUE FOR REVIEW ---
                  content_hash_id           client_hash_id  gsc_impressions  \
8054586  content_eadb33b5df496f4a  client_e547b89c05043229            39305   
9179913  content_eadb33b5df496f4a  client_e547b89c05043229            38436   
8972299  content_eadb33b5df496f4a  client_e547b89c05043229            35404   
8550024  content_44f34c0a90047651  client_23a62021009f63c4            32958   
8865290  content_44f34c0a90047651  client_23a62021009f63c4            32756   
8640840  content_eadb33b5df496f4a  client_e547b89c05043229            34817   
9767993  content_eadb33b5df496f4a  client_e547b89c05043229            34606   
7406798  content_eadb33b5df496f4a  client_e547b89c05043229            33571   
7822469  content_44f34c0a90047651  client_23a62021009f63c4            30964   
7767282  content_44f34c0a90047651  client_23a62021009f63c4            30791   
94

## 3. Top-20 review

**1. Action:** `URGENT_METADATA_REFRESH`. **Code:** `PAGE_1_LOW_CTR_BLEED`. **Confidence:** High (Impressions: 39,305, Pos: 2.20, CTR: 0.64%). **Wrong if:** The query is a zero-click search (e.g., calculator or weather) where Google answers it directly on the SERP.

**2. Action:** `URGENT_METADATA_REFRESH`. **Code:** `PAGE_1_LOW_CTR_BLEED`. **Confidence:** High (Impressions: 38,436, Pos: 2.20, CTR: 0.71%). **Wrong if:** The traffic is driven by navigational brand terms meant for a competitor.

**3. Action:** `URGENT_METADATA_REFRESH`. **Code:** `PAGE_1_LOW_CTR_BLEED`. **Confidence:** High (Impressions: 35,404, Pos: 2.19, CTR: 0.64%). **Wrong if:** It is an off-season landing page generating irrelevant trailing impressions.

**4. Action:** `URGENT_METADATA_REFRESH`. **Code:** `PAGE_1_LOW_CTR_BLEED`. **Confidence:** High (Impressions: 32,958, Pos: 0.13, CTR: 0.00%). **Wrong if:** The page is a gated PDF or image file, which naturally suppresses CTR versus standard HTML articles.

**5. Action:** `URGENT_METADATA_REFRESH`. **Code:** `PAGE_1_LOW_CTR_BLEED`. **Confidence:** High (Impressions: 32,756, Pos: 0.14, CTR: 0.01%). **Wrong if:** Internal GA4 tracking broke for this client, making the denominator/proxy inaccurate.

**6. Action:** `URGENT_METADATA_REFRESH`. **Code:** `PAGE_1_LOW_CTR_BLEED`. **Confidence:** High (Impressions: 34,817, Pos: 2.18, CTR: 0.64%). **Wrong if:** Google recently introduced an AI Overview for this keyword, suppressing organic clicks industry-wide.

**7. Action:** `URGENT_METADATA_REFRESH`. **Code:** `PAGE_1_LOW_CTR_BLEED`. **Confidence:** High (Impressions: 34,606, Pos: 2.24, CTR: 0.68%). **Wrong if:** The page was published yesterday and ranking has not stabilized (Staleness context missing).

**8. Action:** `URGENT_METADATA_REFRESH`. **Code:** `PAGE_1_LOW_CTR_BLEED`. **Confidence:** High (Impressions: 33,571, Pos: 2.31, CTR: 0.64%). **Wrong if:** It is an internal site-search URL indexed by mistake.

**9. Action:** `URGENT_METADATA_REFRESH`. **Code:** `PAGE_1_LOW_CTR_BLEED`. **Confidence:** High (Impressions: 30,964, Pos: 0.12, CTR: 0.00%). **Wrong if:** The URL is a generic Privacy Policy page that holds no commercial value.

**10. Action:** `URGENT_METADATA_REFRESH`. **Code:** `PAGE_1_LOW_CTR_BLEED`.
**Confidence:** High (Impressions: 30,791, Pos: 0.09, CTR: 0.01%). **Wrong if:** The low CTR is strictly mobile-driven, while desktop CTR remains healthy (missing device split).

**11. Action:** `URGENT_METADATA_REFRESH`. **Code:** `PAGE_1_LOW_CTR_BLEED`. **Confidence:** Medium (Impressions: 30,573, Pos: 0.24, CTR: 0.01%). **Wrong if:** Keyword cannibalization is occurring with a stronger internal page that is capturing the actual clicks.

**12. Action:** `URGENT_METADATA_REFRESH`. **Code:** `PAGE_1_LOW_CTR_BLEED`. **Confidence:** Medium (Impressions: 32,665, Pos: 2.39, CTR: 0.68%). **Wrong if:** The page represents an out-of-stock e-commerce product that should be redirected, not refreshed.

**13. Action:** `URGENT_METADATA_REFRESH`. **Code:** `PAGE_1_LOW_CTR_BLEED`. **Confidence:** Medium (Impressions: 32,462, Pos: 2.37, CTR: 0.67%). **Wrong if:** The impressions are artificially inflated by automated bot scrapers querying the SERP.

**14. Action:** `URGENT_METADATA_REFRESH`. **Code:** `PAGE_1_LOW_CTR_BLEED`. **Confidence:** Medium (Impressions: 31,713, Pos: 2.33, CTR: 0.67%). **Wrong if:** The SERP is dominated by Google Image packs, pushing the organic text link visually below the fold despite a "Page 1" position.

**15. Action:** `URGENT_METADATA_REFRESH`. **Code:** `PAGE_1_LOW_CTR_BLEED`. **Confidence:** Medium (Impressions: 31,133, Pos: 2.23, CTR: 0.64%). **Wrong if:** The SERP is dominated by a Local Map Pack (e.g., "near me" searches) siphoning the clicks.

**16. Action:** `URGENT_METADATA_REFRESH`. **Code:** `PAGE_1_LOW_CTR_BLEED`. **Confidence:** Medium (Impressions: 28,527, Pos: 2.40, CTR: 0.68%). **Wrong if:** A breaking news carousel is temporarily burying standard organic results.

**17. Action:** `URGENT_METADATA_REFRESH`. **Code:** `PAGE_1_LOW_CTR_BLEED`. **Confidence:** Medium (Impressions: 25,582, Pos: 2.39, CTR: 0.19%). **Wrong if:** The query is dominated by a video snippet (YouTube) which users click instead of the article.

**18. Action:** `URGENT_METADATA_REFRESH`. **Code:** `PAGE_1_LOW_CTR_BLEED`. **Confidence:** Medium (Impressions: 24,335, Pos: 2.38, CTR: 0.19%). **Wrong if:** The page is ranking via nested sitelinks under the client's homepage, distorting the standard CTR curve.

**19. Action:** `URGENT_METADATA_REFRESH`. **Code:** `PAGE_1_LOW_CTR_BLEED`. **Confidence:** Medium (Impressions: 22,321, Pos: 2.47, CTR: 0.02%). **Wrong if:** The search query is in a foreign language not supported by the page content, causing immediate visual bounce on the SERP.

**20. Action:** `URGENT_METADATA_REFRESH`. **Code:** `PAGE_1_LOW_CTR_BLEED`. **Confidence:** Medium (Impressions: 23,365, Pos: 2.46, CTR: 0.85%). **Wrong if:** The content was hit by a "thin content" penalty mid-month, meaning historical impressions look healthy but current trajectory is dead.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Weak picks + leakage check

## The Concept:
You need to explicitly identify where your baseline rule falls apart (the "weak picks") and programmatically prove that you didn't cheat by including target labels or future data (the "leakage check").

Copy and paste this into the Markdown cell for Section 4:

## Weak Picks Analysis:
The weakest picks in this baseline model are pages resting exactly on the minimum volume threshold (e.g., exactly 100 impressions) with zero clicks. A mathematical CTR of 0.0% heavily penalizes the score, but 100 impressions over 30 days is only ~3 impressions a day. This is statistical noise, not a true bleed. To fix this, a robust ML model will need a higher volume floor or Bayesian smoothing for CTR.

## Leakage Check:
Confirmed clean. No future-window metrics or product flags were utilized. The score relies strictly on historical GSC metrics (trailing impressions, clicks, average position) knowable at the exact moment of decision. The target labels (is_declining_label, trend_direction) were deliberately excluded from the baseline generation script.

In [ ]:
# Programmatically verify no future labels or target variables leaked into the feature set
leaked_columns = ['is_declining_label', 'trend_direction', 'trend_pct']
current_columns = baseline_df.columns.tolist()

leak_found = any(col in current_columns for col in leaked_columns)

print("-" * 50)
if leak_found:
    print("❌ LEAKAGE CHECK FAILED - Target variables found in the baseline frame!")
else:
    print("✅ LEAKAGE CHECK PASSED - No future/label features detected in baseline generation.")
print("-" * 50)

--------------------------------------------------
✅ LEAKAGE CHECK PASSED - No future/label features detected in baseline generation.
--------------------------------------------------


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.